In [34]:
import pandas as pd

df_2022 = pd.read_csv('dados_brutos/faixa_etaria_ibge_2022.csv')


In [36]:
import pandas as pd
import re


# Tratamento do nome do município e código
df_2022['Codigo'] = df_2022['Codigo'].astype(str).str[:6]

# 3. Função para mapear idades específicas do arquivo de 2022
def classificar_faixa_2022(nome_coluna):
    nome_coluna_str = str(nome_coluna).lower()
    
    if 'total' in nome_coluna_str:
        return 'Total'
    
    if 'menos de 1' in nome_coluna_str:
        return '0-3'
    
    # Extrai a idade
    numeros = [int(n) for n in re.findall(r'\d+', nome_coluna_str)]
    if not numeros:
        return None
        
    idade = numeros[0]
    
    if 'mais' in nome_coluna_str or idade >= 64:
        return '64+'
    
    if 0 <= idade <= 3: return '0-3'
    elif 4 <= idade <= 6: return '4-6'
    elif 7 <= idade <= 15: return '7-15'
    elif 16 <= idade <= 17: return '16-17'
    elif 18 <= idade <= 24: return '18-24'
    elif 25 <= idade <= 34: return '25-34'
    elif 35 <= idade <= 39: return '35-39'
    elif 40 <= idade <= 44: return '40-44'
    elif 45 <= idade <= 49: return '45-49'
    elif 50 <= idade <= 54: return '50-54'
    elif 55 <= idade <= 59: return '55-59'
    elif 60 <= idade <= 64: return '60-64'
    
    return None

# Manter apenas as colunas de Código, Município e as de idade
colunas_validas = ['Codigo', 'Municipio'] + [c for c in df_2022.columns if c not in ['Codigo', 'Municipio'] and classificar_faixa_2022(c) is not None]
df_2022 = df_2022[colunas_validas]

# 4. Derreter a tabela (Melt)
df_melt = df_2022.melt(id_vars=['Codigo', 'Municipio'], var_name='Idade_Original', value_name='Populacao')
df_melt['Populacao'] = pd.to_numeric(df_melt['Populacao'].replace('-', 0).replace('X', 0), errors='coerce').fillna(0)
df_melt['Faixa_Agrupada'] = df_melt['Idade_Original'].apply(classificar_faixa_2022)
df_melt = df_melt.dropna(subset=['Faixa_Agrupada'])

# 5. Agrupar e Pivotar
df_final_2022 = df_melt.pivot_table(
    index=['Codigo', 'Municipio'], 
    columns='Faixa_Agrupada', 
    values='Populacao', 
    aggfunc='sum'
).reset_index()

# 6. Forçar a ordem das colunas
ordem_colunas = ['Codigo', 'Municipio', '0-3', '4-6', '7-15', '16-17', '18-24', 
                 '25-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '64+', 'Total']

for col in ordem_colunas:
    if col not in df_final_2022.columns:
        df_final_2022[col] = float('nan')

df_final_2022 = df_final_2022[ordem_colunas]
df_final_2022.columns.name = None # Remove o nome do índice de colunas criado pelo pivot

df_final_2022

,Codigo,Municipio,0-3,4-6,7-15,16-17,18-24,25-34,35-39,40-44,45-49,50-54,55-59,60-64,64+,Total
0,15,Pará,538275,421518,1298790,292959,994826,1306506,650689,604614,490203,419361,346819,235151,641181,8120131
1,150010,Abaetetuba (PA),10459,8071,24556,5591,20296,27807,12928,11467,9435,7836,6170,4075,11912,158188
2,150013,Abel Figueiredo (PA),425,373,1001,202,667,864,473,477,343,356,298,204,568,6136
3,150020,Acará (PA),4278,3372,10659,2510,7275,9183,4384,3956,3172,2863,2317,1489,4523,59023
4,150030,Afuá (PA),3512,2693,7775,1630,5170,5879,2453,2189,1797,1414,1148,771,2075,37765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,150815,Uruará (PA),3343,2474,7447,1591,5224,7030,3474,3136,2491,2093,1855,1164,3009,43558
141,150820,Vigia (PA),2944,2382,7517,1793,5651,8222,4232,3963,3179,2796,2258,1591,4899,50832
142,150830,Viseu (PA),4342,3492,11827,2701,7406,8578,4239,3693,2899,2386,2140,1579,4345,58692
143,150835,Vitória do Xingu (PA),1087,962,2856,608,1783,2510,1155,1130,871,755,638,388,1102,15607
